### Challenge 1 : Organiser des données client

In [12]:
class Noeud:
    def __init__(self, nom):
        self.nom = nom
        self.enfants = {}   # clé → Noeud
        self.clients = []   # ids clients (feuilles)

    def ajouter_client(self, chemin, client_id):
        if len(chemin) == 0:
            self.clients.append(client_id)
        else:
            clef = chemin[0]
            if clef not in self.enfants:
                self.enfants[clef] = Noeud(clef)
            self.enfants[clef].ajouter_client(chemin[1:], client_id)

    def compter_clients(self):
        total = len(self.clients)
        for enfant in self.enfants.values():
            total += enfant.compter_clients()
        return total

    def afficher(self, indent=""):
        print(f"{indent}{self.nom}")
        for enfant in self.enfants.values():
            enfant.afficher(indent + "  ")
        if self.clients:
            print(f"{indent}  Clients {', '.join('Client ' + str(c) for c in self.clients)}")


def groupe_age(age):
    if age < 30:
        return "<30 ans"
    elif 30 <= age <= 40:
        return "30-40 ans"
    else:
        return ">40 ans"


clients = [
    {"id": 1, "age": 25, "ville": "Paris", "departement": "75"},
    {"id": 2, "age": 35, "ville": "Lyon", "departement": "69"},
    {"id": 3, "age": 28, "ville": "Paris", "departement": "75"},
    {"id": 4, "age": 42, "ville": "Marseille", "departement": "13"},
    {"id": 5, "age": 31, "ville": "Lyon", "departement": "69"},
]

racine = Noeud("France")

for client in clients:
    chemin = [client["departement"], client["ville"], groupe_age(client["age"])]
    racine.ajouter_client(chemin, client["id"])

racine.afficher()

France
  75
    Paris
      <30 ans
        Clients Client 1, Client 3
  69
    Lyon
      30-40 ans
        Clients Client 2, Client 5
  13
    Marseille
      >40 ans
        Clients Client 4


## Challenge 4 : Segmentation clients avec arbre

In [16]:
# Création de l'arbre de décision (comme défini précédemment)
class NoeudDecision:
    def __init__(self, nom, condition=None, segment=None):
        self.nom = nom
        self.condition = condition
        self.segment = segment
        self.gauche = None
        self.droite = None

    def classifier(self, montant, frequence):
        if self.segment is not None:
            return self.segment
        if self.condition(montant, frequence):
            return self.gauche.classifier(montant, frequence)
        else:
            return self.droite.classifier(montant, frequence)

# Feuilles
feuille_A = NoeudDecision("Segment A", segment='A')
feuille_B = NoeudDecision("Segment B", segment='B')
feuille_C = NoeudDecision("Segment C", segment='C')

# Branche fréquence > 2 (pour montant > 100)
noeud_freq = NoeudDecision("Fréquence > 2 ?", condition=lambda m, f: f > 2)
noeud_freq.gauche = feuille_A
noeud_freq.droite = feuille_B

# Racine : Montant > 100
racine = NoeudDecision("Montant > 100 ?", condition=lambda m, f: m > 100)
racine.gauche = noeud_freq

# Branche fréquence > 2 (pour montant <= 100)
noeud_freq_non = NoeudDecision("Fréquence > 2 ?", condition=lambda m, f: f > 2)
noeud_freq_non.gauche = feuille_B
noeud_freq_non.droite = feuille_C

racine.droite = noeud_freq_non

# Nouveaux clients
nouveaux_clients = [
    {"client": "Client1", "montant": 90,  "frequence": 4},
    {"client": "Client2", "montant": 110, "frequence": 1},
    {"client": "Client3", "montant": 60,  "frequence": 2},
]

# Classifier clients et afficher résultat
segments = []
for c in nouveaux_clients:
    segment = racine.classifier(c["montant"], c["frequence"])
    print(f"{c['client']} → Segment {segment}")
    segments.append(segment)

# Calcul de la répartition
from collections import Counter
compte = Counter(segments)
print("\nRépartition des segments :")
for seg, count in compte.items():
    print(f"Segment {seg}: {count} client(s)")

Client1 → Segment B
Client2 → Segment B
Client3 → Segment C

Répartition des segments :
Segment B: 2 client(s)
Segment C: 1 client(s)
